# ANALYSE-09 : Un carreau de ℤ³ qui pave sans périodicité totale

**Série** : SymbolicAI / Lean — Digestions de résultats profonds, companion du capstone [Lean-20 — Digestions Tao](../Lean-20-Capstone-Digestions-Tao-Python.ipynb)
**Source** : famille 155 du corpus openai/math (« A counterexample to periodic tiling in dimension three », squelette Lean publié) — gisement de reconnaissance `G:\Mon Drive\MyIA\IA\Distillations\2026-10-openai-math\`
**Lac source** : `sudoku_lean` (la sémantique exact-cover / pavage discret du dépôt, `MyIA.AI.Notebooks/Sudoku/sudoku_lean/`)
**Kernel** : `.net-csharp` — le carnet illustre les définitions et mesure des cas jouets ; l'énoncé formel reste celui du corpus

## Navigation

| Notebook précédent | Notebook suivant |
|---|---|
| [ANALYSE-08 - Ramsey et VdW](ANALYSE-08-Ramsey-VdW.ipynb) | [ANALYSE-10 - Réflexion sur Thurston](ANALYSE-10-Reflexion-Thurston-Python.ipynb) |

**Version Python** : [ANALYSE-09 (Python)](ANALYSE-09-Tuilage-Aperiodique.ipynb)

**Version anglaise** : [ANALYSE-09 (en)](ANALYSE-09-Tuilage-Aperiodique_en.ipynb)

***

## Pourquoi ce carnet

La conjecture du pavage périodique dit, grossièrement : *si une forme pave l'espace, elle le pave périodiquement*. Elle est vraie en dimension 1, vraie pour une tuile en dimension 2 (sur ℤ²), et **fausse à partir de la dimension 3** : la famille 155 du corpus openai/math exhibe un **carreau unique** — une partie finie T ⊂ ℤ³ — qui pave ℤ³ par translations mais dont **aucun pavage n'est totalement périodique**.

C'est le contre-exemple dans la **plus petite dimension possible** pour un carreau unique. L'épaississement en cubes unitaires transporte le même contre-exemple dans ℝ³, même en autorisant des vecteurs de translation réels arbitraires.

> **Statut épistémique** : openai/math est une source tierce non vérifiée par nous. L'énoncé ci-dessus est celui déclaré par le corpus (squelette Lean publié pour la famille 155) ; ce carnet le **présente** et l'**illustre** sur des cas jouets mesurés — il ne re-démontre pas le théorème.

## Ce que le carnet établit (plan)

1. **§1** — les définitions exactes : tuile, pavage par translations, période, *périodicité totale* (rang 3) — et pourquoi « apériodique » est plus subtil que « sans aucune période ».
2. **§2** — la connexion exact-cover : paver une fenêtre finie, c'est résoudre une couverture exacte — la sémantique que le lac `sudoku_lean` formalise.
3. **§3** — le moteur (backtracking borné + mesure du rang des périodes) et quatre expériences mesurées : un témoin périodique, une tuile qui ne pave **aucune** boîte (et pourquoi c'est un théorème), une tuile à période minimale **prouvée** égale à 4, et un piège : une tuile qui **semble** apériodique sur petite fenêtre.
4. **§4** — lecture honnête : ce que les fenêtres finies peuvent et ne peuvent pas établir, avec les mesures du §3 comme preuves.

## 1. Définitions — tuile, pavage, périodicité totale

**Tuile.** Une *tuile* est une partie **finie** non vide T ⊂ ℤ³. Ses éléments sont des cellules ; on pense à un polycube connexe, mais la définition n'exige pas la connexité.

**Pavage par translations.** La tuile T *pave* ℤ³ s'il existe un ensemble de translations S ⊂ ℤ³ tel que les copies T + s (s ∈ S) recouvrent ℤ³ **exactement** : chaque cellule de ℤ³ appartient à une et une seule copie. C'est une **couverture exacte** de ℤ³ par des copies translatées de T — ni trou, ni chevauchement.

**Période, pavage périodique.** Un vecteur v ∈ ℤ³ \ {0} est une *période* d'un pavage si l'ensemble de ses translations est invariant par v : translater tout le pavage de v le laisse inchangé. Le **groupe des périodes** d'un pavage est un sous-groupe de ℤ³ ; il a un **rang** r ∈ {0, 1, 2, 3} (dimension du ℤ-module).

**Périodicité totale.** Un pavage est *totalement périodique* quand r = 3 : trois périodes indépendantes. C'est la notion que le théorème 155 frappe : la tuile contre-exemple pave ℤ³, mais **aucun** de ses pavages n'atteint le rang 3.

**Pourquoi cette nuance compte.** « Apériodique » ne veut pas dire « zéro période » : un pavage peut admettre une ou deux directions de périodes (rang 1 ou 2) et rester un contre-exemple — ce qui est interdit, c'est le rang plein. Un pavage de rang 2, par exemple, se répète dans deux directions mais jamais dans la troisième : assez pour réfuter la conjecture, pas assez pour être « sans structure ».

## 2. La connexion exact-cover — ce que `sudoku_lean` formalise

Sur une **fenêtre finie** W (un bloc N₁×N₂×N₃ de ℤ³), paver W par des copies de T est un problème de couverture exacte : chaque cellule de W doit être couverte **exactement une fois**, chaque placement de tuile est un « choix », et les choix doivent être deux à deux non conflictuants.

C'est exactement la sémantique du lac `sudoku_lean` du dépôt : `ExactCover.IsExactCover` (`MyIA.AI.Notebooks/Sudoku/sudoku_lean/Sudoku/ExactCover.lean`) définit la couverture exacte d'un ensemble de *scopes* par une sélection, et le Sudoku y est l'instance canonique. Le Sudoku est un pavage : chaque case reçoit exactement un chiffre, chaque contrainte est couverte exactement une fois. Notre fenêtre de pavage est une autre instance de la même sémantique — cells × placements au lieu de cells × valeurs.

> **Copie pédagogique déclarée** (règle organ-first du dépôt) : le solveur borné de ce carnet est une réimplémentation locale **déclarée** — backtracking pur C#, lisible — de la sémantique exact-cover. L'organe natif (`sudoku_lean`) est un lac de preuves Lean, pas une librairie .NET consommable ; l'invocation réelle n'est pas disponible côté C#, et le but ici est de montrer la mécanique, pas de la prouver. Réponses aux 5 questions : la série Sudoku possède la sémantique (1) ; son module n'est pas invocable depuis C# (2) ; l'exporter exigerait un port sans valeur pédagogique ici (3) ; le témoin négatif de l'organe est la preuve formelle `mem_toSelection_iff`, hors de portée d'un carnet (4) ; la vérification indépendante est le re-jeu multi-fenêtres du §3, dont les rangs mesurés confrontent les contraintes prouvées à la main (5).

## 3. Le moteur et les quatre expériences

Le moteur a deux organes, tous deux déterministes :

- `PlaceTiles(window, tile)` — backtracking exact-cover **borné** : la première cellule libre (ordre lexicographique) force le choix de l'ancre qui la couvre, on descend, on revient. Le compteur de nœuds mesure le coût combinatoire.
- `PeriodRank(placement, window)` — pour le pavage obtenu, teste tous les vecteurs de période v dans la boîte |vᵢ| ≤ ⌊Nᵢ/2⌋ (la **demi-fenêtre**) et retourne le rang sur ℤ du groupe qu'ils engendrent (élimination gaussienne exacte sur les rationnels).

La demi-fenêtre est la clef de lecture de tout le §3 : **toute période plus longue que la demi-fenêtre est invisible**. C'est un plafond de mesure, pas une propriété de la tuile — les expériences A et B2 le montrent chacun à leur façon.

Quatre expériences :

| # | Tuile | Ce qu'on mesure | Contrainte prouvable à la main |
|---|---|---|---|
| A | domino 1×1×2 | le témoin périodique : rang 3 attendu | pavage évident, périodes 1 ou 2 |
| B0 | prisme L×{0} | la tuile ne pave **aucune** boîte alignée | théorème : coin forcé, induction, case (1,1) morte |
| B1 | domino troué {(0,0,0),(2,0,0)} | rang 2 puis rang 3 quand la fenêtre grandit | période x minimale **exactement 4** dans tout pavage |
| B2 | cube dilaté {0,2}³ | rang 0 puis rang 3 | le motif d'ancres {0,1}³ + 4ℤ³ donne un pavage de groupe de périodes 4ℤ³ ; aucune unicité globale n'est affirmée |

In [1]:
// Moteur de calcul pour le carnet ANALYSE-09-Tuilage-Aperiodique
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

public static class TilingEngine
{
    // Budget de nœuds épuisé : instance abandonnée, ni échec décidé ni succès.
    private sealed class NodeBudgetExceeded : Exception { }

    private static (int X, int Y, int Z) Add((int X, int Y, int Z) a, (int X, int Y, int Z) b)
        => (a.X + b.X, a.Y + b.Y, a.Z + b.Z);

    private static bool Inside((int X, int Y, int Z) p, (int X, int Y, int Z) window)
        => p.X >= 0 && p.X < window.X && p.Y >= 0 && p.Y < window.Y && p.Z >= 0 && p.Z < window.Z;

    // Placement : cellule -> ancre de la copie qui la couvre.
    public static (bool Solved, Dictionary<(int X, int Y, int Z), (int X, int Y, int Z)> Placement, int Nodes)
        PlaceTiles((int X, int Y, int Z) window, IEnumerable<(int X, int Y, int Z)> tile, int maxNodes = 2_000_000)
        => PlaceTiles(window, tile, Array.Empty<(int, int, int)>(), maxNodes);

    // Translations seules, aucune rotation ; une collection vide représente l'absence de trous.
    public static (bool Solved, Dictionary<(int X, int Y, int Z), (int X, int Y, int Z)> Placement, int Nodes)
        PlaceTiles((int X, int Y, int Z) window, IEnumerable<(int X, int Y, int Z)> tile,
            IEnumerable<(int X, int Y, int Z)> forbidden, int maxNodes = 2_000_000)
    {
        var (nx, ny, nz) = window;
        if (nx < 1 || ny < 1 || nz < 1)
            throw new ArgumentOutOfRangeException(nameof(window), "La fenêtre doit valoir au moins 1 sur chaque axe.");

        var cells = tile.Distinct().OrderBy(p => p).ToArray();
        if (cells.Length == 0)
            throw new ArgumentException("La tuile est vide.", nameof(tile));

        // 0 = libre, 1 = couvert, 2 = trou.
        var occ = new byte[checked(nx * ny * nz)];
        int Index((int X, int Y, int Z) p) => (p.X * ny + p.Y) * nz + p.Z;
        int Normalize(int i, int n) => i < 0 ? i + n : i;

        // L'ordre du set Python n'influe pas sur le masque : on fixe néanmoins l'ordre lexicographique.
        foreach (var p in forbidden.OrderBy(p => p))
        {
            var q = (Normalize(p.X, nx), Normalize(p.Y, ny), Normalize(p.Z, nz));
            if (!Inside(q, window))
                throw new ArgumentOutOfRangeException(nameof(forbidden), "Trou hors de la fenêtre.");
            occ[Index(q)] = 2;
        }

        var placement = new Dictionary<(int X, int Y, int Z), (int X, int Y, int Z)>();
        int freeCount = occ.Count(value => value == 0);
        if (freeCount % cells.Length != 0)
            return (false, placement, 1); // volume incompatible avec |T| : échec immédiat, décidé.

        var anchors = new List<(int X, int Y, int Z)>();
        int nodes = 0;

        List<(int X, int Y, int Z)> Candidates((int X, int Y, int Z) c)
        {
            var result = new List<(int X, int Y, int Z)>();
            foreach (var t in cells)
            {
                var a = (c.X - t.X, c.Y - t.Y, c.Z - t.Z);
                if (result.Contains(a)) continue;
                bool valid = true;
                foreach (var u in cells)
                {
                    var p = Add(a, u);
                    if (!Inside(p, window) || occ[Index(p)] != 0)
                    {
                        valid = false;
                        break;
                    }
                }
                if (valid) result.Add(a);
            }
            result.Sort();
            return result;
        }

        bool Solve()
        {
            if (++nodes > maxNodes) throw new NodeBudgetExceeded();
            int k = Array.IndexOf(occ, (byte)0); // première cellule libre en ordre C (x lent, z rapide).
            if (k < 0) return true; // plus aucune cellule libre : succès.
            var c = (k / (ny * nz), (k / nz) % ny, k % nz);
            foreach (var a in Candidates(c))
            {
                var placed = cells.Select(u => Index(Add(a, u))).ToArray();
                foreach (int i in placed) occ[i] = 1;
                anchors.Add(a);
                if (Solve()) return true;
                anchors.RemoveAt(anchors.Count - 1);
                foreach (int i in placed) occ[i] = 0;
            }
            return false;
        }

        try
        {
            if (!Solve()) return (false, placement, nodes);
        }
        catch (NodeBudgetExceeded)
        {
            return (false, placement, nodes);
        }

        foreach (var a in anchors)
            foreach (var u in cells)
                placement[Add(a, u)] = a;
        return (true, placement, nodes);
    }

    public static (int Rank, List<(int X, int Y, int Z)> Generators)
        PeriodRank(IReadOnlyDictionary<(int X, int Y, int Z), (int X, int Y, int Z)> placement,
            (int X, int Y, int Z) window)
    {
        // Cellules présentes dans le placement : les clés tiennent lieu de masque.
        // Le dictionnaire est parcouru explicitement dans l'ordre lexicographique.
        var occupied = placement.OrderBy(entry => entry.Key).ToArray();
        var (nx, ny, nz) = window;
        var periods = new HashSet<(int X, int Y, int Z)>();
        for (int dx = -(nx / 2); dx <= nx / 2; dx++)
        for (int dy = -(ny / 2); dy <= ny / 2; dy++)
        for (int dz = -(nz / 2); dz <= nz / 2; dz++)
        {
            var v = (dx, dy, dz);
            if (v == (0, 0, 0)) continue;
            if (nx - Math.Abs(dx) < 1 || ny - Math.Abs(dy) < 1 || nz - Math.Abs(dz) < 1)
                continue; // chevauchement vide : « période » vide de sens, on ignore.

            bool both = false;
            bool equal = true;
            foreach (var entry in occupied)
            {
                var q = Add(entry.Key, v);
                if (!Inside(q, window) || !placement.TryGetValue(q, out var shifted)) continue;
                both = true;
                if (shifted != Add(entry.Value, v))
                {
                    equal = false;
                    break;
                }
            }
            if (!both) continue; // chevauchement réduit aux seuls trous : période vide de sens.
            if (!equal) continue; // égalité des ancres, axe vec inclus.

            // Forme canonique : premier coefficient non nul positif.
            int first = dx != 0 ? dx : dy != 0 ? dy : dz;
            periods.Add(first < 0 ? (-dx, -dy, -dz) : v);
        }

        // Préférence pour les générateurs courts (norme L1, puis ordre) : pure
        // lisibilité pédagogique, sans effet sur le rang ni le déterminisme.
        var ordered = periods.OrderBy(p => Math.Abs((long)p.X) + Math.Abs((long)p.Y) + Math.Abs((long)p.Z))
            .ThenBy(p => p);
        var generators = IndependentBasis(ordered);
        return (generators.Count, generators);
    }

    private static List<(int X, int Y, int Z)> IndependentBasis(IEnumerable<(int X, int Y, int Z)> vectors)
    {
        var rows = new List<(int Pivot, BigInteger[] Row)>();
        var kept = new List<(int X, int Y, int Z)>();
        foreach (var v in vectors)
        {
            var row = new BigInteger[] { v.X, v.Y, v.Z };
            foreach (var (pivot, previous) in rows)
            {
                BigInteger coefficient = row[pivot];
                if (coefficient.IsZero) continue;

                // Même élimination sur ℚ sans fractions : le pivot multiplie toute la ligne.
                BigInteger scale = previous[pivot];
                for (int j = 0; j < row.Length; j++)
                    row[j] = scale * row[j] - coefficient * previous[j];
            }
            int nextPivot = Array.FindIndex(row, value => !value.IsZero);
            if (nextPivot < 0) continue;
            rows.Add((nextPivot, row));
            kept.Add(v);
        }
        return kept;
    }
}

In [2]:
// Mesurer, comparer et afficher une fenêtre
var comparaisonsParite = new List<(string Experience, bool Conforme)>();

string FormaterCellule((int X, int Y, int Z) p)
    => FormattableString.Invariant($"({p.X}, {p.Y}, {p.Z})");

(string Tile, (int X, int Y, int Z)[] Cells, (int X, int Y, int Z) Window, int Nodes,
    bool Solved, int Rank, IReadOnlyList<(int X, int Y, int Z)> Generators)
    Mesurer(string experience, string nom, IEnumerable<(int X, int Y, int Z)> cells,
        (int X, int Y, int Z) win, string referencePython)
{
    var cellules = cells.Distinct().OrderBy(p => p).ToArray();
    var (solved, placement, nodes) = TilingEngine.PlaceTiles(win, cellules);
    string ligne = FormattableString.Invariant($" {nom} | fenetre {FormaterCellule(win)} | nodes {nodes}");
    int rang = -1; // aucun rang pour une fenêtre non résolue.
    IReadOnlyList<(int X, int Y, int Z)> generateurs = Array.Empty<(int, int, int)>();
    if (solved)
    {
        var periodes = TilingEngine.PeriodRank(placement, win);
        rang = periodes.Rank;
        generateurs = periodes.Generators;
        ligne += FormattableString.Invariant($" | rang {rang} | generateurs [{string.Join(", ", generateurs.Select(p => FormaterCellule(p)))}]");
    }
    else
    {
        ligne += " | NON RESOLU";
    }

    // L'égalité ordinale de la ligne contrôle aussi le nombre de nœuds et l'ordre des générateurs.
    bool conforme = string.Equals(ligne, referencePython, StringComparison.Ordinal);
    comparaisonsParite.Add((experience, conforme));
    if (!conforme)
        throw new InvalidOperationException($"ÉCART {experience} :\nC# calculé :{ligne}\nPython cité :{referencePython}");

    Console.WriteLine(ligne);
    return (nom, cellules, win, nodes, solved, rang, generateurs);
}

### 3.1 Expérience A — le domino, témoin périodique

Deux fenêtres : un cube 4×4×4 (le régime confortable) et une fenêtre volontairement **mince** (6,4,2) — la période (0,0,2) du domino y dépasse la demi-fenêtre z (= 1).

In [3]:
// Illustration A — domino 1×1×2 : témoin périodique
// A -- domino 1x1x2 : controle periodique.
var domino = new HashSet<(int X, int Y, int Z)> { (0, 0, 0), (0, 0, 1) };

// Références citées : sorties enregistrées de la cellule 7 du carnet Python.
const string referenceAConfort = " domino | fenetre (4, 4, 4) | nodes 33 | rang 3 | generateurs [(0, 1, 0), (1, 0, 0), (0, 0, 2)]";
const string referenceAMince = " domino | fenetre (6, 4, 2) | nodes 25 | rang 2 | generateurs [(0, 1, 0), (1, 0, 0)]";
var aConfort = Mesurer("A", "domino", domino, (4, 4, 4), referenceAConfort);
var aMince = Mesurer("A", "domino", domino, (6, 4, 2), referenceAMince);

 domino | fenetre (4, 4, 4) | nodes 33 | rang 3 | generateurs [(0, 1, 0), (1, 0, 0), (0, 0, 2)]


 domino | fenetre (6, 4, 2) | nodes 25 | rang 2 | generateurs [(0, 1, 0), (1, 0, 0)]


### Lecture du résultat

Sur la fenêtre confortable (4,4,4), le groupe des périodes observé est de **rang 3** — trois générateurs indépendants dont (0,0,2) : le domino est totalement périodique, la mesure le voit. Sur la fenêtre mince (6,4,2), le rang mesuré tombe à **2** alors que la tuile n'a pas changé et reste périodique : la période (0,0,2) dépasse la demi-fenêtre z (⌊2/2⌋ = 1) et devient **invisible**. Le témoin périodique échoue donc lui-même à se montrer périodique quand la fenêtre est trop petite — première leçon de mesure du carnet : **une période n'est visible que si chacune de ses coordonnées reste sous la demi-fenêtre**. Ce plafond de visibilité ne se lit pas comme une borne du rang d'un pavage infini : le §4 sépare le cas du motif répétable et celui de la restriction d'un pavage quelconque.

### 3.2 Expérience B0 — le prisme en L : un échec de recherche qui est un théorème

Le triomino en L du plan xy ({(0,0,0), (1,0,0), (0,1,0)}), prismé avec une épaisseur 1 en z. On le lance sur trois fenêtres croissantes — il n'en résout **aucune**, en très peu de nœuds. Ce n'est pas un épuisement du solveur : c'est une **preuve**. Dans chaque plan z (la tuile étant plate), le coin (0,0) d'une boîte n'est couvrable que par l'ancre (0,0) — les deux autres candidats sortent — ce qui couvre (1,0) et (0,1) ; puis (0,2) n'est couvrable que par l'ancre (0,2), qui couvre (1,2) et (0,3) ; par induction toute la colonne 0 et les cases (1, pair) sont couvertes par des ancres forcées, et la case (1,1) n'a plus **aucun** candidat valide. Aucun rectangle 2D n'est tuilable par le L à orientation fixée, et le prisme plat hérite du résultat plan par plan.

Leçon pour la famille 155 : une tuile peut être **si** contraignante qu'elle interdit tout pavage — le contre-exemple 155 est l'équilibre inverse et subtil, une tuile qui pave ℤ³ mais interdit seulement la périodicité totale.

In [4]:
// Illustration B0 — prisme en L dans trois boîtes alignées
// B0 -- prisme L x {0} : ne tuile AUCUNE boite alignee (theoreme, cf. prose).
var lPrisme = new HashSet<(int X, int Y, int Z)> { (0, 0, 0), (1, 0, 0), (0, 1, 0) };

// Références citées : sorties enregistrées de la cellule 10 du carnet Python.
var casB0 = new ((int X, int Y, int Z) Fenetre, string ReferencePython)[]
{
    ((3, 3, 3), " L_prisme | fenetre (3, 3, 3) | nodes 4 | NON RESOLU"),
    ((4, 3, 3), " L_prisme | fenetre (4, 3, 3) | nodes 4 | NON RESOLU"),
    ((6, 4, 2), " L_prisme | fenetre (6, 4, 2) | nodes 5 | NON RESOLU")
};
var b0 = casB0.Select(cas => Mesurer("B0", "L_prisme", lPrisme, cas.Fenetre, cas.ReferencePython)).ToArray();

 L_prisme | fenetre (3, 3, 3) | nodes 4 | NON RESOLU


 L_prisme | fenetre (4, 3, 3) | nodes 4 | NON RESOLU


 L_prisme | fenetre (6, 4, 2) | nodes 5 | NON RESOLU


### 3.3 Expérience B1 — le domino troué : une période minimale prouvée, et l'artefact de fenêtre

Le domino troué {(0,0,0),(2,0,0)} : deux cellules séparées d'un trou. Sur chaque ligne x, chaque cellule est soit une ancre, soit une ancre + 2 (exactement une fois chacune) : notant f(n) ∈ {0,1} l'indicateur « n est une ancre », on a **f(n) + f(n−2) = 1 pour tout n**. La phase des ancres est donc exactement 4-périodique, et **aucune période purement axiale (a, 0, 0) avec 0 < a < 4 n'existe dans aucun pavage** — une contrainte prouvée à la main, pas mesurée. (Une période dont la composante x est petite peut exister si elle change aussi de ligne : ainsi (1, 1, 0) est une période du pavage d'ancres {(x, y, z) : x − y ≡ 0 ou 1 (mod 4)}.) Conséquence immédiate sur la mesure : une fenêtre de demi-largeur < 4 ne peut jamais certifier la période axiale x. On mesure (4,2,2) puis (8,2,2).

### 3.4 Expérience B2 — le cube dilaté : le piège « semble apériodique »

Le cube dilaté {0,2}³ : les 8 coins d'un cube 2×2×2 espacé. Chaque copie ne couvre que des cellules de **sa classe de parité** (ajouter 2 préserve la parité de chaque coordonnée). L'ensemble d'ancres **{0,1}³ + 4ℤ³** fournit un pavage explicite de ℤ³ : dans chaque coordonnée, les ancres congrues à 0 ou 1 modulo 4 et leurs décalages de 2 couvrent chaque entier exactement une fois. Le groupe des périodes de **ce motif** est exactement 4ℤ³.

Il ne s'agit pas d'un pavage unique : par exemple, sa translation d'une unité selon x donne un autre ensemble d'ancres. La décomposition en classes de parité ne démontre pas non plus une unicité à translation près. L'affirmation d'unicité du carnet Python n'est donc pas retenue ici ; le commentaire du code décrit le motif témoin.

Sur (4,4,4), aucune période de ce motif n'entre dans la boîte de test (|vᵢ| ≤ 2) : le rang mesuré tombe à **0** — l'allure « totalement apériodique » — alors que son groupe de périodes infini est de rang 3. Sur (8,8,8), les trois générateurs (4,0,0), (0,4,0), (0,0,4) apparaissent.

In [5]:
// Illustrations B1 et B2 — domino troué et cube dilaté
// B1 -- domino troue : periode x minimale PROUVEE = 4 (cf. prose 3.3).
var dominoTroue = new HashSet<(int X, int Y, int Z)> { (0, 0, 0), (2, 0, 0) };

// Références citées : sorties enregistrées de la cellule 12 du carnet Python.
const string referenceB1Etroite = " domino_troue | fenetre (4, 2, 2) | nodes 9 | rang 2 | generateurs [(0, 0, 1), (0, 1, 0)]";
const string referenceB1Large = " domino_troue | fenetre (8, 2, 2) | nodes 17 | rang 3 | generateurs [(0, 0, 1), (0, 1, 0), (4, 0, 0)]";
var b1Etroite = Mesurer("B1", "domino_troue", dominoTroue, (4, 2, 2), referenceB1Etroite);
var b1Large = Mesurer("B1", "domino_troue", dominoTroue, (8, 2, 2), referenceB1Large);

// B2 -- cube dilate : motif temoin d'ancres {0,1}^3 + 4Z^3, de periodes exactement 4Z^3 (cf. prose 3.4).
// Ce motif n'est pas l'unique pavage : sa translation d'une unite selon x donne un autre ensemble d'ancres ;
// la mesure ne porte que sur le motif trouve par le moteur.
var cubeDilate = new HashSet<(int X, int Y, int Z)>();
var coordonneesCube = new[] { 0, 2 };
foreach (int x in coordonneesCube)
foreach (int y in coordonneesCube)
foreach (int z in coordonneesCube)
    cubeDilate.Add((x, y, z));

const string referenceB2Petite = " cube_dilate | fenetre (4, 4, 4) | nodes 9 | rang 0 | generateurs []";
const string referenceB2Grande = " cube_dilate | fenetre (8, 8, 8) | nodes 65 | rang 3 | generateurs [(0, 0, 4), (0, 4, 0), (4, 0, 0)]";
var b2Petite = Mesurer("B2", "cube_dilate", cubeDilate, (4, 4, 4), referenceB2Petite);
var b2Grande = Mesurer("B2", "cube_dilate", cubeDilate, (8, 8, 8), referenceB2Grande);

// Le bilan agrège les comparaisons exécutées et vérifie qu'aucune fenêtre ne manque.
var couvertureAttendue = new (string Experience, int Nombre)[] { ("A", 2), ("B0", 3), ("B1", 2), ("B2", 2) };
if (comparaisonsParite.Count != couvertureAttendue.Sum(cas => cas.Nombre))
    throw new InvalidOperationException("Nombre total de comparaisons de parité incorrect.");

Console.WriteLine("Bilan de parité — C# calculé / Python cité (égalité exacte)");
foreach (var (experience, attendu) in couvertureAttendue)
{
    var verifications = comparaisonsParite.Where(cas => cas.Experience == experience).ToArray();
    int ecarts = verifications.Count(cas => !cas.Conforme);
    if (verifications.Length != attendu || ecarts != 0)
        throw new InvalidOperationException(FormattableString.Invariant($"ÉCART {experience} : {verifications.Length}/{attendu} lignes comparées, {ecarts} écart(s)."));
    Console.WriteLine(FormattableString.Invariant($"{experience} | {verifications.Length}/{attendu} lignes comparées | {ecarts} écart | CONFORME"));
}


 domino_troue | fenetre (4, 2, 2) | nodes 9 | rang 2 | generateurs [(0, 0, 1), (0, 1, 0)]


 domino_troue | fenetre (8, 2, 2) | nodes 17 | rang 3 | generateurs [(0, 0, 1), (0, 1, 0), (4, 0, 0)]


 cube_dilate | fenetre (4, 4, 4) | nodes 9 | rang 0 | generateurs []


 cube_dilate | fenetre (8, 8, 8) | nodes 65 | rang 3 | generateurs [(0, 0, 4), (0, 4, 0), (4, 0, 0)]


Bilan de parité — C# calculé / Python cité (égalité exacte)


A | 2/2 lignes comparées | 0 écart | CONFORME


B0 | 3/3 lignes comparées | 0 écart | CONFORME


B1 | 2/2 lignes comparées | 0 écart | CONFORME


B2 | 2/2 lignes comparées | 0 écart | CONFORME


### Lecture du résultat

Le domino troué : rang **2** sur (4,2,2) — les périodes partielles (0,1,0) et (0,0,1) seulement — puis rang **3** sur (8,2,2) où le générateur (4,0,0) devient visible, exactement la période minimale prouvée. Le cube dilaté pousse le même phénomène à l'extrême : rang **0** sur (4,4,4) — l'allure la plus apériodique qui soit — puis rang **3** sur (8,8,8) avec les trois générateurs de 4ℤ³. Les motifs infinis explicites considérés pour ces deux tuiles sont **totalement périodiques** ; leurs groupes de périodes sont de rang 3, mais les petites fenêtres n'en révèlent pas toutes les directions.

- petite fenêtre ⇒ certaines périodes du motif infini peuvent rester invisibles ⇒ rang mesuré plus petit, comme en B1 et B2 ;
- un rang 3 détecté par `PeriodRank` établit trois invariances indépendantes **sur les chevauchements testés de la fenêtre**, pas à lui seul la périodicité totale d'un pavage infini donné ;
- dans ces exemples, la périodicité totale des motifs infinis découle de leur construction explicite. Un rang < 3 sur une fenêtre n'est pas un certificat d'apériodicité.

## 4. Ce qu'une fenêtre finie peut établir — et ce qu'elle ne peut pas

Une fenêtre finie ne peut **jamais prouver** l'apériodicité, pour une raison simple : un pavage d'une boîte W par des tuiles **entièrement contenues** dans W se prolonge en un pavage de ℤ³ totalement périodique — répéter le contenu de W par les translations de la fenêtre. Les mesures du §3 le portent avec les chiffres :

- le **témoin** A lui-même rend rang 2 sur une fenêtre mince — le plafond de demi-fenêtre frappe même les tuiles les plus simples ;
- le domino troué (B1) n'admet, dans aucun pavage, de période (a, 0, 0) avec 0 < a < 4 (contrainte prouvée au §3.3), et le cube dilaté (B2) admet le motif d'ancres {0,1}³ + 4ℤ³, de groupe de périodes exactement 4ℤ³ : leurs rangs mesurés 2 et 0 sur petites fenêtres sont des **artefacts de visibilité**, levés quand la fenêtre double ;
- le prisme en L (B0) rappelle l'autre bord du spectre : une contrainte locale peut interdire **tout** pavage d'une boîte — l'échec du solveur y est une preuve, parce qu'elle est courte et structurée (ancres forcées), pas un épuisement.

**Le rang lu sur une fenêtre est une propriété du contenu de la fenêtre, pas d'un pavage infini.** Deux cas sont à séparer :

- **Des tuiles entièrement contenues pavent la boîte — le cas du moteur.** Le contenu de W est un **motif répétable** : le rang mesuré décrit les symétries de ce motif, et le motif répété donne un pavage totalement périodique. Il ne dit rien des autres pavages de la même tuile : la même fenêtre peut aussi être la restriction d'un pavage infini de rang plus faible.
- **La fenêtre est la restriction d'un pavage infini quelconque — des tuiles peuvent traverser le bord.** Le rang observé n'est **pas un minorant** du rang de ce pavage. Contre-exemple, sur le témoin A : un pavage par dominos verticaux garde la même phase sur toutes les colonnes de la fenêtre, et le calcul y rend rang 3. Changeons la phase d'une seule colonne hors fenêtre : ce pavage infini n'est plus invariant que sous les translations verticales paires (0, 0, 2k), et son groupe de périodes est de rang 1.

Ce que la mesure apporte donc : **elle calibre la visibilité** (une période n'entre dans la boîte de test que si chacune de ses coordonnées reste sous la demi-fenêtre), **elle compare** (le coût en nœuds du backtracking dit si la contrainte locale mord), **elle piège** (B2 montre qu'un rang 0 mesuré ne dit rien du vrai groupe). Ce qu'elle ne donne pas : une borne sur le rang d'un pavage infini. Un rang < 3 n'est pas un certificat d'apériodicité ; un rang 3 n'est pas, à lui seul, un certificat de périodicité totale. Ce qu'elle ne remplace pas : le théorème 155, qui établit l'absence de rang 3 pour **tous** les pavages de sa tuile — un quantificateur universel qu'aucune fenêtre finie n'atteint.

C'est la division du travail de la série ANALYSE : le carnet C# **mesure et illustre**, la formalisation **prouve** — ici, le squelette Lean publié par le corpus pour la famille 155.

## 5. Exercices

### Exercice 1 — le groupe des périodes, pas seulement son rang

La fonction `PeriodRank` rend le rang et des générateurs. Écrivez `PeriodGroup(placement, windowShape)` qui retourne la **liste complète** des vecteurs de période trouvés (sous forme canonique), puis déterminez sur le domino le pas minimal dans chaque direction et justifiez le résultat.

### Exercice 2 — la plus petite fenêtre qui révèle le générateur x

Trouvez expérimentalement la plus petite fenêtre (N₁,2,2) à partir de laquelle le générateur axial x du domino troué devient **visible** dans la mesure. Que se passe-t-il pour N₁ = 7 ?

### Exercice 3 — construire une contrainte de parité

Construisez une autre tuile pour laquelle les **classes de parité** permettent de construire un pavage de ℤ³ et d'en prouver l'**unicité**, en précisant la notion d'unicité visée et les éventuelles conditions de normalisation : l'unicité littérale sans normalisation est exclue par les translations, comme expliqué au §3.4. Démontrez cette unicité dans le cadre annoncé. Déterminez les rangs et les générateurs attendus sur deux fenêtres de tailles différentes, puis vérifiez-les expérimentalement avec le moteur.

## Conclusion

### Ce qui a été établi

| Résultat | Mesure |
|---|---|
| Le domino est totalement périodique | rang 3 sur (4,4,4), générateurs dont (0,0,2) |
| Une fenêtre mince cache même les périodes évidentes | rang 2 sur (6,4,2) pour ce même domino |
| Le prisme en L ne pave aucune boîte alignée | 0/3 fenêtres résolues, échec court et structuré (théorème) |
| Le domino troué a une période x minimale prouvée de 4 | rang 2 sur (4,2,2) puis 3 sur (8,2,2), générateur (4,0,0) |
| Le cube dilaté admet le motif d'ancres {0,1}³ + 4ℤ³, de groupe de périodes 4ℤ³ | rang 0 sur (4,4,4) puis 3 sur (8,8,8) ; aucune unicité globale n'est affirmée |
| Une fenêtre finie ne prouve jamais l'apériodicité | argument du §4 : une boîte pavée par des tuiles contenues se répète en un pavage totalement périodique |
| Le rang lu sur une fenêtre ne borne pas le rang d'un pavage infini | contre-exemple du §4 : dominos verticaux, rang 3 lu sur A, rang 1 pour le pavage prolongé |

**Portée du rang mesuré.** Quand des tuiles entièrement contenues pavent la boîte — le cas du moteur —, le contenu de la boîte est un **motif répétable** : le rang mesuré décrit ce motif, et non les autres pavages de la même tuile, dont la même fenêtre peut être la restriction. Quand la fenêtre est la restriction d'un pavage infini quelconque, des tuiles pouvant traverser le bord, le rang observé n'est pas un minorant du rang de ce pavage. Dans les deux cas, un rang lu sur une fenêtre n'est ni un certificat d'apériodicité, ni un certificat de périodicité totale.

### Ce qui reste au théorème

L'énoncé 155 — un carreau fini de ℤ³ qui pave sans **aucun** pavage de rang 3 — n'est pas re-démontré ici : il est **présenté**, sourcé du corpus openai/math (squelette Lean publié), et illustré par des cas jouets mesurés dont chacun borne honnêtement sa portée. Le pont naturel est double : la formalisation côté Lean (le squelette du corpus), et l'exact-cover côté `sudoku_lean` — la sémantique est la même, seules les instances changent.

**Pistes pour la suite** : le passage ℝ³ (épaississement en cubes unitaires, translations réelles arbitraires) ; la comparaison avec les jeux de tuiles apériodiques classiques (Berger 1966, Jeandel–Rao en 2D — une *famille* de tuiles, là où 155 donne un carreau *unique*) ; et la complexité algorithmique du pavage (indécidabilité en général, le résultat de Berger).

> Ce carnet reprend le [jumeau Python](ANALYSE-09-Tuilage-Aperiodique.ipynb). Le bilan de parité en fin de §3.4 compare les mesures C# aux sorties Python citées.